# Phase 1: Project Setup and Data Acquisition

## Task 1.1 Configure Development Environment

### Deliverables

- Anaconda environment
- Required libraries installed
- Project structure verified

### Section 1: Environment Verification

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import sqlite3
from pathlib import Path
from pymongo import MongoClient

print(pd.__version__)
print(np.__version__)

3.0.5
2.4.6


### Section 2: Project Structure Validation

In [3]:
project_root = Path.cwd().parent

folders = [
    "data/telco",
    "data/retail_churn",
    "data/retail_sales",
    "notebooks",
    "sqlite",
    "reports"
]

for folder in folders:
    print(Path(project_root, folder).exists(), folder)

True data/telco
True data/retail_churn
True data/retail_sales
True notebooks
True sqlite
True reports


## Task 1.2 Verify Dataset Structure

### Section 3: Discover Dataset Files

In [4]:
for root, dirs, files in os.walk("../data"):
    for file in files:
        print(os.path.join(root, file))

../data\processed\clean_retail_churn.csv
../data\processed\clean_retail_sales.csv
../data\processed\clean_telco.csv
../data\retail_churn\churn_retail.xlsx
../data\retail_sales\retail_data.csv
../data\telco\Telco_customer_churn.xlsx


### Section 4: Load Each Dataset

In [5]:
# %conda install -c conda-forge zstandard

In [6]:
telco_df = pd.read_excel("../data/telco/Telco_customer_churn.xlsx")
retail_churn_df = pd.read_excel("../data/retail_churn/churn_retail.xlsx", sheet_name="E Comm")
retail_sales_df = pd.io.parsers.read_csv("../data/retail_sales/retail_data.csv")

### Section 5: Basic Dataset Summary

In [7]:
print(telco_df.shape)
print(telco_df.columns.tolist())

(7043, 33)
['CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude', 'Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn Label', 'Churn Value', 'Churn Score', 'CLTV', 'Churn Reason']


In [8]:
print(retail_churn_df.shape)
print(retail_churn_df.columns.tolist())

(5630, 20)
['CustomerID', 'Churn', 'Tenure', 'PreferredLoginDevice', 'CityTier', 'WarehouseToHome', 'PreferredPaymentMode', 'Gender', 'HourSpendOnApp', 'NumberOfDeviceRegistered', 'PreferedOrderCat', 'SatisfactionScore', 'MaritalStatus', 'NumberOfAddress', 'Complain', 'OrderAmountHikeFromlastYear', 'CouponUsed', 'OrderCount', 'DaySinceLastOrder', 'CashbackAmount']


In [9]:
print(retail_sales_df.shape)
print(retail_sales_df.columns.tolist())

(1000000, 78)
['customer_id', 'age', 'gender', 'income_bracket', 'loyalty_program', 'membership_years', 'churned', 'marital_status', 'number_of_children', 'education_level', 'occupation', 'transaction_id', 'transaction_date', 'product_id', 'product_category', 'quantity', 'unit_price', 'discount_applied', 'payment_method', 'store_location', 'transaction_hour', 'day_of_week', 'week_of_year', 'month_of_year', 'avg_purchase_value', 'purchase_frequency', 'last_purchase_date', 'avg_discount_used', 'preferred_store', 'online_purchases', 'in_store_purchases', 'avg_items_per_transaction', 'avg_transaction_value', 'total_returned_items', 'total_returned_value', 'total_sales', 'total_transactions', 'total_items_purchased', 'total_discounts_received', 'avg_spent_per_category', 'max_single_purchase_value', 'min_single_purchase_value', 'product_name', 'product_brand', 'product_rating', 'product_review_count', 'product_stock', 'product_return_rate', 'product_size', 'product_weight', 'product_color', 

## Task 1.3 Create Data Dictionary

### Section 6: Create Initial Data Dictionary

In [10]:
# Convert to Numeric
telco_df["Total Charges"] = pd.to_numeric(telco_df["Total Charges"], errors="coerce")

In [11]:
telco_dict = pd.DataFrame({
    "Column": telco_df.columns,
    "DataType": telco_df.dtypes.astype(str),
    "Nullable": telco_df.isnull().any().values,
    "UniqueValues": [telco_df[col].nunique() for col in telco_df.columns]
})

retail_churn_dict = pd.DataFrame({
    "Column": retail_churn_df.columns,
    "DataType": retail_churn_df.dtypes.astype(str),
    "Nullable": retail_churn_df.isnull().any().values,
    "UniqueValues": [retail_churn_df[col].nunique() for col in retail_churn_df.columns]
})

retail_sales_dict = pd.DataFrame({
    "Column": retail_sales_df.columns,
    "DataType": retail_sales_df.dtypes.astype(str),
    "Nullable": retail_sales_df.isnull().any().values,
    "UniqueValues": [retail_sales_df[col].nunique() for col in retail_sales_df.columns]
})

In [12]:
telco_dict.to_csv(
    "../reports/telco_data_dictionary.csv",
    index=False
)

retail_churn_dict.to_csv(
    "../reports/retail_churn_data_dictionary.csv",
    index=False
)

retail_sales_dict.to_csv(
    "../reports/retail_sales_data_dictionary.csv",
    index=False
)

### Section 7: Project Feasibility Assessment

In [13]:
# Dataset metadata
inventory = [
    {
        "name": "Telco Customer Churn",
        "file": "../data/telco/Telco_customer_churn.xlsx",
        "df": telco_df,
        "target": "Churn Label"
    },
    {
        "name": "Retail Customer Churn",
        "file": "../data/retail_churn/churn_retail.xlsx",
        "df": retail_churn_df,
        "target": "Churn"
    },
    {
        "name": "Retail Sales",
        "file": "../data/retail_sales/retail_data.csv",
        "df": retail_sales_df,
        "target": "churned"
    }
]

md = []
md.append("# Dataset Inventory\n")
md.append("## Project Datasets\n")

for ds in inventory:
    df = ds["df"]

    md.append(f"### {ds['name']}\n")
    md.append(f"- **Source File:** `{ds['file']}`")
    md.append(f"- **Rows:** {df.shape,}")
    md.append(f"- **Columns:** {df.shape[1]}")
    md.append(f"- **Target Variable:** {ds['target']}")
    md.append(f"- **Missing Values:** {int(df.isnull().sum().sum())}")
    md.append("")
    md.append("#### Columns")
    md.append("")

    for col, dtype in zip(df.columns, df.dtypes):
        md.append(f"- `{col}` ({dtype})")

    md.append("")
    md.append("---")
    md.append("")

# Write markdown file
output_path = Path("../reports/dataset_inventory.md")

with open(output_path, "w", encoding="utf-8") as f:
    f.write("\n".join(md))

print(f"Dataset inventory created: {output_path.resolve()}")

Dataset inventory created: C:\Users\sicnarf00\OneDrive\GSU\CIS8005\PyCharmWorkspace\Project\reports\dataset_inventory.md


# Phase 2: Data Profiling and Quality Assessment

## Task 2.1 Perform Data Profiling

In [14]:
# %%capture cell_output
print("Data Profiling Telco Churn")
telco_df.info()
telco_df.describe()
telco_df.head()

Data Profiling Telco Churn
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   str    
 1   Count              7043 non-null   int64  
 2   Country            7043 non-null   str    
 3   State              7043 non-null   str    
 4   City               7043 non-null   str    
 5   Zip Code           7043 non-null   int64  
 6   Lat Long           7043 non-null   str    
 7   Latitude           7043 non-null   float64
 8   Longitude          7043 non-null   float64
 9   Gender             7043 non-null   str    
 10  Senior Citizen     7043 non-null   str    
 11  Partner            7043 non-null   str    
 12  Dependents         7043 non-null   str    
 13  Tenure Months      7043 non-null   int64  
 14  Phone Service      7043 non-null   str    
 15  Multiple Lines     7043 non-null   str    
 16  Internet

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.50,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,Yes,1,89,5340,Competitor had better devices


In [15]:
with open("../reports/telco_profile.txt", "w") as f:
    f.write(cell_output.stdout)

NameError: name 'cell_output' is not defined

In [ ]:
%%capture cell_output
print("Data Profiling Retail Churn")
retail_churn_df.info()
retail_churn_df.describe()
retail_churn_df.head()

In [ ]:
with open("../reports/retail_churn_profile.txt", "w") as f:
    f.write(cell_output.stdout)

In [ ]:
%%capture cell_output
print("Data Profiling Retail Sales")
retail_sales_df.info()
retail_sales_df.describe()
retail_sales_df.head()

In [ ]:
with open("../reports/retail_sales_profile.txt", "w") as f:
    f.write(cell_output.stdout)

### Missing Value Analysis

In [ ]:
%%capture cell_output
print("Missing Value Analysis Telco")
missing = pd.DataFrame({
    "MissingCount": telco_df.isnull().sum(),
    "MissingPercent":
        telco_df.isnull().mean() * 100
})

print(missing.sort_values(
    "MissingPercent",
    ascending=False
))

In [ ]:
with open("../reports/telco_missing_value_analysis.txt", "w") as f:
    f.write(cell_output.stdout)

In [ ]:
%%capture cell_output
print("Missing Value Analysis Retail")
missing = pd.DataFrame({
    "MissingCount": retail_churn_df.isnull().sum(),
    "MissingPercent":
        retail_churn_df.isnull().mean() * 100
})

print(missing.sort_values(
    "MissingPercent",
    ascending=False
))

In [ ]:
with open("../reports/retail_churn_missing_value_analysis.txt", "w") as f:
    f.write(cell_output.stdout)

In [ ]:
%%capture cell_output
print("Missing Value Analysis Retail Sales")
missing = pd.DataFrame({
    "MissingCount": retail_sales_df.isnull().sum(),
    "MissingPercent":
        retail_sales_df.isnull().mean() * 100
})

print(missing.sort_values(
    "MissingPercent",
    ascending=False
))

In [ ]:
with open("../reports/retail_sales_missing_value_analysis.txt", "w") as f:
    f.write(cell_output.stdout)

### Duplicate Analysis

In [ ]:
%%capture cell_output
print("Duplicate Analysis Telco")
print("telco_df.duplicated().sum():",telco_df.duplicated().sum())
print('telco_df["CustomerID"].duplicated().sum():',telco_df["CustomerID"].duplicated().sum())

In [ ]:
with open("../reports/telco_duplicate_analysis.txt", "w") as f:
    f.write(cell_output.stdout)

In [ ]:
%%capture cell_output
print("Duplicate Analysis Retail")
print("retail_churn_df.duplicated().sum():",retail_churn_df.duplicated().sum())
print('retail_churn_df["CustomerID"].duplicated().sum()',retail_churn_df["CustomerID"].duplicated().sum())

In [ ]:
with open("../reports/retail_churn_duplicate_analysis.txt", "w") as f:
    f.write(cell_output.stdout)

In [ ]:
%%capture cell_output
print("Duplicate Analysis Retail Sales")
print('retail_sales_df.duplicated().sum():',retail_sales_df.duplicated().sum())
print('retail_sales_df["customer_id"].duplicated().sum():',retail_sales_df["customer_id"].duplicated().sum())

In [ ]:
with open("../reports/retail_sales_duplicate_analysis.txt", "w") as f:
    f.write(cell_output.stdout)

### Unique Value Analysis

In [ ]:
%%capture cell_output
print("Unique Value Analysis Telco")
unique_values = pd.DataFrame({
    "UniqueValues":
        telco_df.nunique()
})
print(unique_values)

In [ ]:
with open("../reports/telco_unique_value_analysis.txt", "w") as f:
    f.write(cell_output.stdout)

In [ ]:
%%capture cell_output
print("Unique Value Analysis Retail")
unique_values = pd.DataFrame({
    "UniqueValues":
        retail_churn_df.nunique()
})
print(unique_values)

In [ ]:
with open("../reports/retail_churn_unique_value_analysis.txt", "w") as f:
    f.write(cell_output.stdout)

In [ ]:
%%capture cell_output
print("Unique Value Analysis Retail Sales")
unique_values = pd.DataFrame({
    "UniqueValues":
        retail_sales_df.nunique()
})
print(unique_values)

In [ ]:
with open("../reports/retail_sales_unique_value_analysis.txt", "w") as f:
    f.write(cell_output.stdout)

### Numerical Statistics

In [ ]:
%%capture cell_output
print("Numerical Statistics Telco")
print(telco_df.describe().T)

In [ ]:
with open("../reports/telco_numerical_statistics.txt", "w") as f:
    f.write(cell_output.stdout)

In [ ]:
%%capture cell_output
print("Numerical Statistics Retail")
print(retail_churn_df.describe().T)

In [ ]:
with open("../reports/retail_churn_numerical_statistics.txt", "w") as f:
    f.write(cell_output.stdout)

In [ ]:
%%capture cell_output
print("Numerical Statistics Retail Sales")
print(retail_sales_df.describe().T)

In [ ]:
with open("../reports/retail_sales_numerical_statistics.txt", "w") as f:
    f.write(cell_output.stdout)

### Categorical Analysis

In [ ]:
%%capture cell_output
print("Categorical Analysis Telco")
for col in telco_df.select_dtypes(
    include="object"
):
    print(f"\n{col}")
    print(
        telco_df[col].value_counts().head(10)
    )

In [ ]:
with open("../reports/telco_categorical_analysis.txt", "w") as f:
    f.write(cell_output.stdout)

In [ ]:
%%capture cell_output
print("Categorical Analysis Retail")
for col in retail_churn_df.select_dtypes(
    include="object"
):
    print(f"\n{col}")
    print(
        retail_churn_df[col].value_counts().head(10)
    )

In [ ]:
with open("../reports/retail_churn_categorical_analysis.txt", "w") as f:
    f.write(cell_output.stdout)

In [ ]:
%%capture cell_output
print("Categorical Analysis Retail Sales")
for col in retail_sales_df.select_dtypes(
    include="object"
):
    print(f"\n{col}")
    print(
        retail_sales_df[col].value_counts().head(10)
    )

In [ ]:
with open("../reports/retail_sales_categorical_analysis.txt", "w") as f:
    f.write(cell_output.stdout)

### Date Profiling

In [ ]:
%%capture cell_output
print("Date Profiling Retail Sales")
date_columns = [
    "transaction_date",
    "last_purchase_date",
    "product_manufacture_date",
    "product_expiry_date",
    "promotion_start_date",
    "promotion_end_date"
]
print(retail_sales_df[date_columns].dtypes)

print("Sample Date Values")
for col in date_columns:
    print(f"\n{col}")
    print(retail_sales_df[col].head())

print("Invalid Date Detection")
for col in date_columns:

    temp = pd.to_datetime(
        retail_sales_df[col],
        errors="coerce"
    )

    invalid = temp.isna().sum()

    print(
        col,
        "Invalid Dates:",
        invalid
    )

print("Date Coverage")
for col in date_columns:

    temp = pd.to_datetime(
        retail_sales_df[col],
        errors="coerce"
    )

    print(
        col,
        temp.min(),
        temp.max()
    )

In [ ]:
with open("../reports/retail_sales_date_profiling.txt", "w") as f:
    f.write(cell_output.stdout)

### Outlier Detection

#### Telco Outlier Candidates

In [ ]:
numeric_candidates = [
    "Tenure Months",
    "Monthly Charges",
    "Total Charges",
    "CLTV"
]

for col in numeric_candidates:
    telco_df[col] = pd.to_numeric(
        telco_df[col],
        errors="coerce"
    )

numeric_cols = telco_df.select_dtypes(
    include=["int64", "float64"]
).columns

telco_outlier_summary = []

for col in numeric_cols:

    Q1 = telco_df[col].quantile(0.25)
    Q3 = telco_df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = telco_df[
        (telco_df[col] < lower) |
        (telco_df[col] > upper)
    ]

    telco_outlier_summary.append({
        "Column": col,
        "OutlierCount": len(outliers),
        "OutlierPercent":
            round(
                len(outliers) /
                len(telco_df) * 100,
                2
            )
    })

telco_outlier_df = pd.DataFrame(
    telco_outlier_summary
).sort_values(
    "OutlierCount",
    ascending=False
)

telco_outlier_df.to_csv(
    "../reports/telco_outliers.csv",
    index=False
)

#### Retail Churn Outlier Candidates

In [ ]:
numeric_candidates = [
    "CustomerID",
    "CityTier",
    "Churn",
    "Complain"
]

for col in numeric_candidates:
    retail_churn_df[col] = pd.to_numeric(
        retail_churn_df[col],
        errors="coerce"
    )

numeric_cols = retail_churn_df.select_dtypes(
    include=["int64", "float64"]
).columns

retail_churn_outlier_summary = []

for col in numeric_cols:

    Q1 = retail_churn_df[col].quantile(0.25)
    Q3 = retail_churn_df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = retail_churn_df[
        (retail_churn_df[col] < lower) |
        (retail_churn_df[col] > upper)
    ]

    retail_churn_outlier_summary.append({
        "Column": col,
        "OutlierCount": len(outliers),
        "OutlierPercent":
            round(
                len(outliers) /
                len(retail_churn_df) * 100,
                2
            )
    })

retail_churn_outlier_df = pd.DataFrame(
    retail_churn_outlier_summary
).sort_values(
    "OutlierCount",
    ascending=False
)

retail_churn_outlier_df.to_csv(
    "../reports/retail_churn_outliers.csv",
    index=False
)

#### Retail Sales Outlier Candidates

In [ ]:
exclude = [
    "customer_id",
    "transaction_id",
    "product_id",
    "promotion_id",
    "customer_zip_code",
    "store_zip_code"
]

numeric_cols = [
    col
    for col in retail_sales_df
        .select_dtypes(include=["int64","float64"])
        .columns
    if col not in exclude
]

outlier_summary = []

for col in numeric_cols:

    Q1 = retail_sales_df[col].quantile(0.25)
    Q3 = retail_sales_df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = retail_sales_df[
        (retail_sales_df[col] < lower) |
        (retail_sales_df[col] > upper)
    ]

    outlier_summary.append({
        "Column": col,
        "OutlierCount": len(outliers),
        "OutlierPercent":
            round(
                len(outliers) /
                len(retail_sales_df) * 100,
                2
            )
    })

outlier_df = pd.DataFrame(outlier_summary)

outlier_df.to_csv(
    "../reports/retail_sales_outliers.csv",
    index=False
)

### Target Variable Analysis

In [ ]:
%%capture cell_output
print("Telco Churn Distribution")
print(
    telco_df["Churn Label"]
    .value_counts()
)

print("\nTelco Churn Percentage")
print(
    telco_df["Churn Label"]
    .value_counts(normalize=True)
    * 100
)

In [ ]:
with open("../reports/telco_target_variable_analysis.txt", "w") as f:
    f.write(cell_output.stdout)

In [ ]:
%%capture cell_output
print("Retail Churn Distribution")
print(
    retail_churn_df["Churn"]
    .value_counts()
)

print("\nRetail Churn Percentage")
print(
    retail_churn_df["Churn"]
    .value_counts(normalize=True)
    * 100
)

In [ ]:
with open("../reports/retail_churn_target_variable_analysis.txt", "w") as f:
    f.write(cell_output.stdout)

In [ ]:
%%capture cell_output
print("Retail Sales Distribution")
print(
    retail_sales_df["churned"]
    .value_counts()
)

print("\nRetail Sales Percentage")
print(
    retail_sales_df["churned"]
    .value_counts(normalize=True)
    * 100
)

In [ ]:
with open("../reports/retail_sales_target_variable_analysis.txt", "w") as f:
    f.write(cell_output.stdout)

## Task 2.2 Build Data Quality Report

In [ ]:
quality_report = pd.DataFrame({
    "Column": telco_df.columns,
    "DataType": telco_df.dtypes.astype(str),
    "MissingCount":
        telco_df.isnull().sum().values,
    "MissingPercent":
        (
            telco_df.isnull().mean() * 100
        ).round(2).values,
    "UniqueValues":
        telco_df.nunique().values
})
quality_report.to_csv(
    "../reports/telco_quality_report.csv",
    index=False
)

In [ ]:
quality_report = pd.DataFrame({
    "Column": retail_churn_df.columns,
    "DataType": retail_churn_df.dtypes.astype(str),
    "MissingCount":
        retail_churn_df.isnull().sum().values,
    "MissingPercent":
        (
            retail_churn_df.isnull().mean() * 100
        ).round(2).values,
    "UniqueValues":
        retail_churn_df.nunique().values
})
quality_report.to_csv(
    "../reports/retail_churn_quality_report.csv",
    index=False
)

In [ ]:
quality_report = pd.DataFrame({
    "Column": retail_sales_df.columns,
    "DataType": retail_sales_df.dtypes.astype(str),
    "MissingCount":
        retail_sales_df.isnull().sum().values,
    "MissingPercent":
        (
            retail_sales_df.isnull().mean() * 100
        ).round(2).values,
    "UniqueValues":
        retail_sales_df.nunique().values
})
quality_report.to_csv(
    "../reports/retail_sales_quality_report.csv",
    index=False
)

### Create Profiling Visualizations

#### Missing Values Bar Chart

In [ ]:
missing = pd.DataFrame({
    "MissingCount": telco_df.isnull().sum(),
    "MissingPercent":
        telco_df.isnull().mean() * 100
})
missing["MissingCount"].plot(
    kind="bar",
    figsize=(10,5)
)

In [ ]:
missing = pd.DataFrame({
    "MissingCount": retail_churn_df.isnull().sum(),
    "MissingPercent":
        retail_churn_df.isnull().mean() * 100
})
missing["MissingCount"].plot(
    kind="bar",
    figsize=(10,5)
)

In [ ]:
missing = pd.DataFrame({
    "MissingCount": retail_sales_df.isnull().sum(),
    "MissingPercent":
        retail_sales_df.isnull().mean() * 100
})
missing["MissingCount"].plot(
    kind="bar",
    figsize=(10,5)
)

#### Churn Distribution

In [ ]:
telco_df["Churn Label"].value_counts().plot(
    kind="bar"
)

In [ ]:
retail_churn_df["Churn"].value_counts().plot(
    kind="bar"
)

#### Monthly Charges Distribution

In [ ]:
telco_df["Monthly Charges"].hist(
    bins=30
)

#### Retail Complaints Distribution

In [ ]:
retail_churn_df[
    "Complain"
].hist()

### Create Profiling Summary

In [ ]:
report_path = Path("../reports/profiling_summary.md")

with open(report_path, "w", encoding="utf-8") as f:

    f.write("# Data Profiling Summary\n\n")
    f.write("## Project Overview\n\n")
    f.write("This report summarizes the initial data profiling and quality assessment for the datasets used in the Customer Churn Analytics project.\n\n")

    datasets = [
        ("Telco Customer Churn", telco_df),
        ("Retail Customer Churn", retail_churn_df),
        ("Retail Sales and Customer Behavior", retail_sales_df)
    ]

    for dataset_name, df in datasets:

        f.write(f"## {dataset_name}\n\n")

        # Basic metrics
        rows, cols = df.shape
        missing_total = int(df.isnull().sum().sum())
        duplicate_rows = int(df.duplicated().sum())

        numeric_cols = len(
            df.select_dtypes(
                include=["int64", "float64"]
            ).columns
        )

        categorical_cols = len(
            df.select_dtypes(
                include=["object", "string", "category"]
            ).columns
        )

        f.write(f"### Dataset Overview\n\n")
        f.write(f"- Rows: {rows:,}\n")
        f.write(f"- Columns: {cols}\n")
        f.write(f"- Numeric Columns: {numeric_cols}\n")
        f.write(f"- Categorical Columns: {categorical_cols}\n")
        f.write(f"- Missing Values: {missing_total:,}\n")
        f.write(f"- Duplicate Rows: {duplicate_rows:,}\n\n")

        # Missing values
        missing = (
            df.isnull()
            .sum()
            .reset_index()
        )

        missing.columns = [
            "Column",
            "MissingCount"
        ]

        missing = missing[
            missing["MissingCount"] > 0
        ].sort_values(
            "MissingCount",
            ascending=False
        )

        f.write("### Missing Values\n\n")

        if len(missing) == 0:
            f.write("- No missing values detected.\n\n")
        else:

            for _, row in missing.iterrows():
                f.write(
                    f"- {row['Column']}: "
                    f"{row['MissingCount']} missing values\n"
                )

            f.write("\n")

        f.write(
            "### Data Quality Issues\n\n"
        )

        issues_found = False

        if missing_total > 0:
            issues_found = True

            f.write(
                f"- {missing_total:,} missing values detected.\n"
            )

        if duplicate_rows > 0:
            issues_found = True

            f.write(
                f"- {duplicate_rows:,} duplicate rows detected.\n"
            )

        if not issues_found:

            f.write(
                "- No major data quality issues detected.\n"
            )

        f.write("\n")
        
        # Datatype summary
        f.write("### Data Type Summary\n\n")

        dtype_counts = (
            df.dtypes.astype(str)
            .value_counts()
        )

        for dtype, count in dtype_counts.items():
            f.write(f"- {dtype}: {count} columns\n")

        f.write("\n")

        # Date Profiling
        f.write("### Date Profiling\n\n")

        date_columns = []

        known_date_columns = [
            "transaction_date",
            "last_purchase_date",
            "product_manufacture_date",
            "product_expiry_date",
            "promotion_start_date",
            "promotion_end_date"
        ]

        for col in known_date_columns:
            if col in df.columns:
                date_columns.append(col)

        if len(date_columns) == 0:

            f.write(
                "- No date columns identified.\n\n"
            )

        else:

            for col in date_columns:

                test_dates = pd.to_datetime(
                    df[col],
                    errors="coerce"
                )

                invalid_dates = (
                    test_dates.isna().sum()
                )

                f.write(
                    f"- {col}: "
                    f"{invalid_dates:,} invalid dates\n"
                )

            f.write("\n")

        # Potential key columns
        f.write("### Potential Key Columns\n\n")

        unique_cols = []

        for col in df.columns:
            if df[col].nunique() == len(df):
                unique_cols.append(col)

        if unique_cols:
            for col in unique_cols:
                f.write(f"- {col}\n")
        else:
            f.write("- No obvious unique identifier detected.\n")

        f.write("\n")

        # Target variable analysis
        f.write(
            "### Target Variable Analysis\n\n"
        )

        target_column = None

        if dataset_name == "Telco Customer Churn":
            target_column = "Churn Label"

        elif dataset_name == "Retail Customer Churn":
            target_column = "Churn"

        elif dataset_name == "Retail Sales and Customer Behavior":
            target_column = "churned"

        if (
            target_column is not None
            and target_column in df.columns
        ):

            target_dist = (
                df[target_column]
                .value_counts(
                    dropna=False
                )
            )

            for value, count in target_dist.items():

                pct = (
                    count /
                    len(df)
                ) * 100

                f.write(
                    f"- {value}: "
                    f"{count:,} "
                    f"({pct:.2f}%)\n"
                )

        else:

            f.write(
                "- Target variable not identified.\n"
            )

        f.write("\n")

        # Profiling observations
        f.write("### Profiling Observations\n\n")

        observations = []

        if missing_total > 0:
            observations.append(
                "Contains missing values that will require treatment during data cleaning."
            )

        if duplicate_rows > 0:
            observations.append(
                "Contains duplicate records that should be investigated."
            )

        object_cols = len(
            df.select_dtypes(
                include=["object", "string"]
            ).columns
        )

        if object_cols > 0:
            observations.append(
                "Contains categorical fields that may require encoding for machine learning."
            )

        if numeric_cols > 0:
            observations.append(
                "Contains numeric attributes that should be evaluated for outliers."
            )

        if len(observations) == 0:
            observations.append(
                "No major profiling issues identified."
            )

        for obs in observations:
            f.write(f"- {obs}\n")

        f.write("\n")
        
        f.write(
            "### Feature Engineering Candidates\n\n"
        )

        if dataset_name == "Telco Customer Churn":

            candidates = [
                "Tenure Months",
                "Monthly Charges",
                "Total Charges",
                "CLTV"
            ]

        elif dataset_name == "Retail Customer Churn":

            candidates = [
                "OrderCount",
                "CouponUsed",
                "CashbackAmount",
                "Tenure",
                "HourSpendOnApp"
            ]

        else:

            candidates = [
                "total_sales",
                "total_transactions",
                "online_purchases",
                "in_store_purchases",
                "customer_support_calls",
                "days_since_last_purchase"
            ]

        for col in candidates:

            if col in df.columns:

                f.write(
                    f"- {col}\n"
                )

    f.write("\n---\n\n")
print(f"Created: {report_path}")